# Leaflet cluster map of talk locations

Assuming you are working in a Linux or Windows Subsystem for Linux environment, you may need to install some dependencies. Assuming a clean installation, the following will be needed:

```bash
sudo apt install jupyter
sudo apt install python3-pip
pip install python-frontmatter getorg --upgrade
```

After which you can run this from the `_talks/` directory, via:

```bash
 jupyter nbconvert --to notebook --execute talkmap.ipynb --output talkmap_out.ipynb
```
 
The `_talks/` directory contains `.md` files of all your talks. This scrapes the location YAML field from each `.md` file, geolocates it with `geopy/Nominatim`, and uses the `getorg` library to output data, HTML, and Javascript for a standalone cluster map.

In [2]:
!pip3 install getorg --upgrade
import glob
import getorg
from geopy import Photon

In [3]:
g = glob.glob("_talks/*.md")

In [4]:
geocoder = Photon(user_agent='http')
location_dict = {}
location = ""
permalink = ""
title = ""

In [11]:

for file in g:
    with open(file, 'r') as f:
        lines = f.read()
        if lines.find('location: "') > 1:
            loc_start = lines.find('location: "') + 11
            lines_trim = lines[loc_start:]
            loc_end = lines_trim.find('"')
            location = lines_trim[:loc_end]
                            
           
        location_dict[location] = geocoder.geocode(location, timeout=10)
        print(location, "\n", location_dict[location])


University of Warwick, Coventry, UK 
 Hitex UK, Viscount Centre 2, CV4 7HS, Viscount Centre 2, Coventry, England, United Kingdom
University of Warwick, Coventry, UK 
 Hitex UK, Viscount Centre 2, CV4 7HS, Viscount Centre 2, Coventry, England, United Kingdom
H. H. Wills Physics Laboratory, University of Bristol, Royal Fort, Bristol, BS8 1TL, UK 
 None
Colorado Convention Center, Denver, CO, USA 
 Colorado Convention Center, 700, 14th Street, 80202, 14th Street, Denver, CO, United States
Manchester Central Convention Centre, Manchester, UK 
 Manchester Central Convention Complex, Windmill Street, M2 3GX, Windmill Street, Manchester, England, United Kingdom
The Bristol Hotel, Prince Street, Bristol, BS1 4QF, UK 
 The Bristol Hotel, Prince Street, BS1 4QF, Prince Street, Bristol, England, United Kingdom
Crowne Plaza Manchester City Centre, Manchester, UK 
 Crowne Plaza, 70, Shudehill, M4 4AF, Shudehill, Manchester, England, United Kingdom
Oak Ridge National Laboratory, Oak Ridge, Tennessee

In [14]:
# Perform geolocation
for file in g:
    # Read the file
    # data = frontmatter.load(file)
    data = file.to_dict()

    # Press on if the location is not present
    if 'location' not in data:
        continue

    # Prepare the description
    title = data['title'].strip()
    venue = data['venue'].strip()
    location = data['location'].strip()
    description = f"{title}<br />{venue}; {location}"

    # Geocode the location and report the status
    try:
        location_dict[description] = geocoder.geocode(location, timeout=TIMEOUT)
        print(description, location_dict[description])
    except ValueError as ex:
        print(f"Error: geocode failed on input {location} with message {ex}")
    except GeocoderTimedOut as ex:
        print(f"Error: geocode timed out on input {location} with message {ex}")
    except Exception as ex:
        print(f"An unhandled exception occurred while processing input {location} with message {ex}")

AttributeError: 'str' object has no attribute 'to_dict'

In [14]:
# Save the map
m = getorg.orgmap.create_map_obj()
getorg.orgmap.output_html_cluster_map(location_dict, folder_name="talkmap", hashed_usernames=False)

'Written map to talkmap/'